In [1]:
import re
import pandas as pd
import time
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns


### Reading in the Data

In [2]:
race = 'TOR330'

In [3]:
TOR330_dem = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_dem.xlsx')

In [4]:
TOR330_dem['Bib'][(TOR330_dem['Year'] == 2022) &
               (TOR330_dem['Retired'] == 'Bosses')].nunique()

0

In [5]:
TOR330_dem.groupby( ['Year','Status1'] )['Status1' ].count()

Year  Status1                     
2021  DNF                             286
      Finished at Courmayeur          431
2022  DNF                             363
      Finished at Bosses               88
      Finished at Courmayeur          408
      Finished at Rifugio Frassati    101
2023  DNF                             585
      Finished at Courmayeur          622
2024  DNF                             563
      Finished at Courmayeur          533
Name: Status1, dtype: int64

In [6]:
lifebase_cut_offs_df = pd.read_excel(f'{race} Data/4. TOR330 Timetable Data/{race}_lifebase_cut_offs_df.xlsx')

# # # Convert integer seconds to timedelta
lifebase_cut_offs_df['Lifebase Duration'] = pd.to_timedelta(lifebase_cut_offs_df['Lifebase Duration_seconds'], unit='s')

# # # Convert integer seconds to timedelta
lifebase_cut_offs_df['Running Total Lifebase Duration'] = pd.to_timedelta(lifebase_cut_offs_df['Running Total Lifebase Duration_seconds'], unit='s')

lifebase_cut_offs_df

,Lifebase,Stage,Lifebase Distance (km),Lifebase Accumulated Distance Elevation (m),Lifebase Elevation Gain (m),Lifebase Accumulated Elevation (m),Lifebase Duration_seconds,Running Total Lifebase Duration_seconds,Lifebase Duration,Running Total Lifebase Duration
0,Valgrisenche IN,Stage 1,48.55,48.55,4339,4339,68400.0,68400,0 days 19:00:00,0 days 19:00:00
1,Valgrisenche OUT,Time Spent in Valgrisenche,0.00,48.55,0,4339,7200.0,75600,0 days 02:00:00,0 days 21:00:00
2,Cogne IN,Stage 2,55.45,104.00,4943,9282,75600.0,151200,0 days 21:00:00,1 days 18:00:00
3,Cogne OUT,Time Spent in Cogne,0.00,104.00,0,9282,7200.0,158400,0 days 02:00:00,1 days 20:00:00
4,Donnas IN,Stage 3,45.77,149.77,2768,12050,64800.0,223200,0 days 18:00:00,2 days 14:00:00
5,Donnas OUT,Time Spent in Donnas,0.00,149.77,0,12050,7200.0,230400,0 days 02:00:00,2 days 16:00:00
6,Gressoney IN,Stage 4,54.23,204.00,5933,17983,75600.0,306000,0 days 21:00:00,3 days 13:00:00
7,Gressoney OUT,Time Spent in Gressoney,0.00,204.00,0,17983,7200.0,313200,0 days 02:00:00,3 days 15:00:00
8,Valtournenche IN,Stage 5,33.62,237.62,3094,21077,64800.0,378000,0 days 18:00:00,4 days 09:00:00
9,Valtournenche OUT,Time Spent in Valtournenche,0.00,237.62,0,21077,7200.0,385200,0 days 02:00:00,4 days 11:00:00


In [7]:
checkpoints_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_checkpoints_bib_df.xlsx')

In [8]:
all_aid_station_bib_df = pd.read_excel(f'TOR330 Data/5. Clean Data for Data Visualisation/TOR330_all_aid_station_bib_df.xlsx')

In [22]:
lifebase_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_lifebase_bib_df.xlsx')

In [23]:
lifebase_bib_df = lifebase_bib_df.merge(
        TOR330_dem[['PK', 'Name','Status1']],
        on=['PK'],
        how='left')
lifebase_bib_df.head()

,PK,Year,Race,Bib,Wave,Lifebase,Stage,Timestamp,Duration_seconds,Running Total Duration_seconds,Banking Time_seconds,Missed Lifebase Allocated Time,Name,Status1
0,TOR330_2021_1,2021,TOR330,1,No Wave in 2021,Start,Stage 1,2021-09-12 10:00:00,NaN,0.0,NaN,Within Allocated Time,Colle Franco,Finished at Courmayeur
1,TOR330_2021_1,2021,TOR330,1,No Wave in 2021,Valgrisenche IN,Stage 1,2021-09-12 17:01:28,25288.0,25288.0,43112.0,Within Allocated Time,Colle Franco,Finished at Courmayeur
2,TOR330_2021_1,2021,TOR330,1,No Wave in 2021,Valgrisenche OUT,Time Spent in Valgrisenche,2021-09-12 17:02:49,81.0,25369.0,50231.0,Within Allocated Time,Colle Franco,Finished at Courmayeur
3,TOR330_2021_1,2021,TOR330,1,No Wave in 2021,Cogne IN,Stage 2,2021-09-13 02:55:10,35541.0,60910.0,90290.0,Within Allocated Time,Colle Franco,Finished at Courmayeur
4,TOR330_2021_1,2021,TOR330,1,No Wave in 2021,Cogne OUT,Time Spent in Cogne,2021-09-13 02:55:54,44.0,60954.0,97446.0,Within Allocated Time,Colle Franco,Finished at Courmayeur


In [32]:
# no_banking_df = lifebase_bib_df[(lifebase_bib_df['Status1'].str.contains( 'Courmayeur')) &
#                 (lifebase_bib_df['Lifebase'] != 'START') &
#                 (lifebase_bib_df['Banking Time_seconds'] <0)]

# no_banking_df['Banking Time_seconds'] = no_banking_df['Banking Time_seconds'].astype(int)
# # Apply function to the column
# no_banking_df['Banking Time'] =pd.to_timedelta(no_banking_df['Banking Time_seconds'], unit='s').astype(str)
no_banking_df['Running Total Duration'] =pd.to_timedelta(no_banking_df['Running Total Duration_seconds'], unit='s').astype(str)

for unique_name in no_banking_df['Name']:
    print(no_banking_df[['Name','Year', 'Race', 'Lifebase','Running Total Duration', 'Banking Time']][no_banking_df['Name'] == unique_name])
    print('*'*40)



C:\Users\Karina\AppData\Local\Temp\ipykernel_1020\2895392653.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_banking_df['Running Total Duration'] =pd.to_timedelta(no_banking_df['Running Total Duration_seconds'], unit='s').astype(str)


             Name  Year    Race           Lifebase Running Total Duration  \
4595  Cirla Marco  2021  TOR330           Cogne IN        1 days 18:59:19   
4596  Cirla Marco  2021  TOR330          Cogne OUT        1 days 21:44:32   
4600  Cirla Marco  2021  TOR330      Gressoney OUT        3 days 16:03:30   
4602  Cirla Marco  2021  TOR330  Valtournenche OUT        4 days 11:38:56   
4603  Cirla Marco  2021  TOR330        Ollomont IN        5 days 08:13:28   

           Banking Time  
4595  -1 days +23:00:41  
4596  -1 days +22:15:28  
4600  -1 days +22:56:30  
4602  -1 days +23:21:04  
4603  -1 days +22:46:32  
****************************************
             Name  Year    Race           Lifebase Running Total Duration  \
4595  Cirla Marco  2021  TOR330           Cogne IN        1 days 18:59:19   
4596  Cirla Marco  2021  TOR330          Cogne OUT        1 days 21:44:32   
4600  Cirla Marco  2021  TOR330      Gressoney OUT        3 days 16:03:30   
4602  Cirla Marco  2021  TOR330 

              Name  Year    Race       Lifebase Running Total Duration  \
5380  Cagna Andrea  2021  TOR330      Cogne OUT        1 days 20:15:59   
5383  Cagna Andrea  2021  TOR330   Gressoney IN        3 days 13:20:27   
5384  Cagna Andrea  2021  TOR330  Gressoney OUT        3 days 16:53:27   

           Banking Time  
5380  -1 days +23:44:01  
5383  -1 days +23:39:33  
5384  -1 days +22:06:33  
****************************************
              Name  Year    Race       Lifebase Running Total Duration  \
5380  Cagna Andrea  2021  TOR330      Cogne OUT        1 days 20:15:59   
5383  Cagna Andrea  2021  TOR330   Gressoney IN        3 days 13:20:27   
5384  Cagna Andrea  2021  TOR330  Gressoney OUT        3 days 16:53:27   

           Banking Time  
5380  -1 days +23:44:01  
5383  -1 days +23:39:33  
5384  -1 days +22:06:33  
****************************************
                  Name  Year    Race       Lifebase Running Total Duration  \
5394  Zanzottera Guido  2021  TOR330  

                              Name  Year    Race       Lifebase  \
5720  Carmona Marina Jose Fernando  2021  TOR330  Gressoney OUT   
5724  Carmona Marina Jose Fernando  2021  TOR330   Ollomont OUT   

     Running Total Duration       Banking Time  
5720        3 days 16:08:22  -1 days +22:51:38  
5724        5 days 09:45:16  -1 days +23:14:44  
****************************************
                  Name  Year    Race     Lifebase Running Total Duration  \
5737   Billi Francesca  2021  TOR330  Ollomont IN        5 days 07:02:30   
47795  Billi Francesca  2024  TOR330       FINISH        6 days 06:53:20   

            Banking Time  
5737   -1 days +23:57:30  
47795  -1 days +23:06:40  
****************************************
                    Name  Year    Race      Lifebase Running Total Duration  \
5752  Bianchetto Stefano  2021  TOR330  Ollomont OUT        5 days 09:41:35   

           Banking Time  
5752  -1 days +23:18:25  
****************************************
       

                Name  Year    Race      Lifebase Running Total Duration  \
5933  Manoni Giorgio  2021  TOR330   Ollomont IN        5 days 08:13:34   
5934  Manoni Giorgio  2021  TOR330  Ollomont OUT        5 days 10:34:29   
5935  Manoni Giorgio  2021  TOR330        FINISH        6 days 06:27:41   

           Banking Time  
5933  -1 days +22:46:26  
5934  -1 days +22:25:31  
5935  -1 days +23:32:19  
****************************************
                 Name  Year    Race      Lifebase Running Total Duration  \
5947  Roques Frederic  2021  TOR330   Ollomont IN        5 days 07:32:52   
5948  Roques Frederic  2021  TOR330  Ollomont OUT        5 days 09:31:43   
5949  Roques Frederic  2021  TOR330        FINISH        6 days 06:29:49   

           Banking Time  
5947  -1 days +23:27:08  
5948  -1 days +23:28:17  
5949  -1 days +23:30:11  
****************************************
                 Name  Year    Race      Lifebase Running Total Duration  \
5947  Roques Frederic  2021 

                 Name  Year    Race       Lifebase Running Total Duration  \
5985  Pierrick Daniel  2021  TOR330   Gressoney IN        3 days 13:11:37   
5986  Pierrick Daniel  2021  TOR330  Gressoney OUT        3 days 16:57:16   
5989  Pierrick Daniel  2021  TOR330    Ollomont IN        5 days 08:15:47   
5990  Pierrick Daniel  2021  TOR330   Ollomont OUT        5 days 10:50:34   
5991  Pierrick Daniel  2021  TOR330         FINISH        6 days 07:09:42   

           Banking Time  
5985  -1 days +23:48:23  
5986  -1 days +22:02:44  
5989  -1 days +22:44:13  
5990  -1 days +22:09:26  
5991  -1 days +22:50:18  
****************************************
                 Name  Year    Race       Lifebase Running Total Duration  \
5985  Pierrick Daniel  2021  TOR330   Gressoney IN        3 days 13:11:37   
5986  Pierrick Daniel  2021  TOR330  Gressoney OUT        3 days 16:57:16   
5989  Pierrick Daniel  2021  TOR330    Ollomont IN        5 days 08:15:47   
5990  Pierrick Daniel  2021  TOR

              Name  Year    Race Lifebase Running Total Duration  \
47781  Xi Hongpeng  2024  TOR330   FINISH        6 days 06:21:09   

            Banking Time  
47781  -1 days +23:38:51  
****************************************
                  Name  Year    Race     Lifebase Running Total Duration  \
5737   Billi Francesca  2021  TOR330  Ollomont IN        5 days 07:02:30   
47795  Billi Francesca  2024  TOR330       FINISH        6 days 06:53:20   

            Banking Time  
5737   -1 days +23:57:30  
47795  -1 days +23:06:40  
****************************************
             Name  Year    Race Lifebase Running Total Duration  \
47809  Lu Sheming  2024  TOR330   FINISH        6 days 06:57:12   

            Banking Time  
47809  -1 days +23:02:48  
****************************************
               Name  Year    Race Lifebase Running Total Duration  \
47823  Zhang Bochao  2024  TOR330   FINISH        6 days 06:59:52   

            Banking Time  
47823  -1 days +23:00

In [ ]:
# # making duration hours 
# datasets = [checkpoints_bib_df, lifebase_bib_df, all_aid_station_bib_df,  TOR330_dem]
# for df in datasets:
#     df['Duration_hours'] = df['Duration_seconds']/ 3600 

In [ ]:
all_aid_station_bib_df

### Getting Average and Median time for lifebases

In [ ]:
Stage1 = [ 'Start', 'Baite Youlaz', 'La Thuile', 'Rifugio Deffeyes',\
          'Planaval', 'Valgrisenche IN']
Stage2 = [ 'Valgrisenche OUT', 'Chalet Epee',\
          'Rhemes-Notre-Dame', 'Eaux Rousse', 'Rifugio Sella', 'Cogne IN']
Stage3 = [  'Cogne OUT', 'Goilles', 'Rifugio Dondena', 'Chardonney', 'Pontboset','Donnas IN']
Stage4 = [  'Donnas OUT', 'Perloz', 'Sassa', 'Rifugio Coda', \
          'Rifugio della Barma', 'Lago Chiaro', 'Col della Vecchia',\
          'Niel La Gruba', 'Loo', 'Gressoney IN']
Stage5 = [  'Gressoney OUT', 'Rifugio Alpenzu', 'Champoluc' ,\
          'Rifugio Grand Tournalin', 'Valtournenche IN']
Stage6 = [   'Valtournenche OUT', 'Rifugio Barmasse', 'Vareton',\
          'Rifugio Magià', 'Rifugio Cuney', 'Bivacco R. Clermont', 'Oyace', \
          'Bruson Arp', 'Ollomont IN']    
Stage7 = [ 'Ollomont OUT', 'Rifugio Champillon', 'Ponteille Desot',\
          'Bosses', 'Rifugio Frassati', 'Pas Entre Deux Sauts',\
          'Monte de la Saxe', 'FINISH']


stages =[ Stage1, Stage2, Stage3, Stage4, Stage5, Stage6, Stage7]
stages_str =[ 'Stage 1', 'Stage 2', 'Stage 3', 'Stage 4', 'Stage 5', 'Stage 6', 'Stage 7']
lifebase_time_spent = ['Valgrisenche OUT','Cogne OUT','Donnas OUT','Gressoney OUT','Valtournenche OUT','Ollomont OUT']


In [ ]:
def creating_time_stats(df, column, category_order):
    
    df_merge = df.merge(
        TOR330_dem[['PK', 'Finish Category']],
        on=['PK'],
        how='left')


    df_merge['Duration_seconds'][df_merge['Duration_seconds']  <= 0] = np.nan

    stats_df = df_merge.groupby(['Finish Category', column])['Duration_seconds'].describe().reset_index(drop = False)


    stats_df[[ 'mean', 'std', 'min', '25%',
           '50%', '75%', 'max']] = stats_df[[ 'mean', 'std', 'min', '25%',
           '50%', '75%', 'max']].round(0)

    # Set 'Finish Category' as a categorical column with the defined order
    stats_df[column] = pd.Categorical(
        stats_df[column],
        categories=category_order,
        ordered=True)
    
    stats_df = stats_df.sort_values(by=column, ascending = True)
    
    

    for finish_category in df_merge['Finish Category'].unique():
        print(finish_category)
        stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'running_total_mean_seconds'
            ] =     stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'mean'
            ].cumsum()


        stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'running_total_median_seconds'
            ] =     stats_df.loc[
                (stats_df['Finish Category'] == finish_category), '50%'
            ].cumsum()


    stats_df = stats_df[['Finish Category', column,
            'count', 'mean', '50%', 'std', 'min', 'max', 
            'running_total_mean_seconds', 'running_total_median_seconds']]


    stats_df = stats_df.rename(columns={'count': f'Count_Finish_Category_{column}_seconds',
                                      'mean': f'Mean_Finish_Category_{column}_seconds',
                                      'std': f'STD_Finish_Category{column}t_seconds',
                                      '50%': f'Median_Finish_Category_{column}_seconds',
                                      'min': f'Min_Finish_Category_{column}_seconds', 
                                      'max': f'Max_Finish_Category_{column}_seconds'})
    stats_df = stats_df[stats_df[column] != 'Start']

    
    
    
    for stage,  stage_str in zip(stages, stages_str):
        stats_df.loc[stats_df[column].isin(stage), 'Stage'] = f'{stage_str}'
        
    for lifebase in lifebase_time_spent:
        lifebase_split = lifebase.split(' OUT')[0] 
        stats_df.loc[stats_df[column] == lifebase, 'Stage'] = f'Time Spent in {lifebase_split}'


    stats_df.to_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_{column}_duration_mean_median_stats_df.xlsx', index = False)

#     print(stats_df[stats_df['Finish Category'] == 'Sub-130'])
    return stats_df

In [ ]:
lifebase_stats_df = creating_time_stats(df = lifebase_bib_df, 
                    column = 'Lifebase', 
                    category_order = ['Start','Valgrisenche IN','Valgrisenche OUT',
                            'Cogne IN',  'Cogne OUT',
                            'Donnas IN', 'Donnas OUT', 
                            'Gressoney IN','Gressoney OUT',
                            'Valtournenche IN','Valtournenche OUT', 
                            'Ollomont IN','Ollomont OUT',
                            'FINISH'])

lifebase_stats_df

In [ ]:
# # # Convert integer seconds to timedelta
lifebase_stats_df['Median_Finish_Category_Lifebase'] = pd.to_timedelta(lifebase_stats_df['Median_Finish_Category_Lifebase_seconds'], unit='s')


lifebase_stats_df[['Finish Category','Lifebase', 'Median_Finish_Category_Lifebase']][lifebase_stats_df['Finish Category'] == 'Sub-130']

In [ ]:
checkpoints_stats_df = creating_time_stats(df = checkpoints_bib_df, 
                    column = 'Checkpoint', 
                    category_order = ['La Thuile', 'Valgrisenche IN', 'Valgrisenche OUT',
                                       'Eaux Rousse', 'Cogne IN', 'Cogne OUT', 'Donnas IN', 'Donnas OUT',
                                       'Rifugio della Barma', 'Niel La Gruba', 'Gressoney IN',
                                       'Gressoney OUT', 'Champoluc', 'Valtournenche IN',
                                       'Valtournenche OUT', 'Oyace', 'Ollomont IN', 'Ollomont OUT',
                                       'FINISH'])

checkpoints_stats_df

In [ ]:
# all_aid_station_bib_stats_df = creating_time_stats(df = all_aid_station_bib_df, 
#                     column = 'Aid Station', 
#                     category_order = ['Baite Youlaz', 'La Thuile', 'Rifugio Deffeyes',
#                                    'Planaval', 'Valgrisenche IN', 'Valgrisenche OUT', 'Chalet Epee',
#                                    'Rhemes-Notre-Dame', 'Eaux Rousse', 'Rifugio Sella', 'Cogne IN',
#                                    'Cogne OUT', 'Goilles', 'Rifugio Dondena', 'Chardonney', 'Pontboset',
#                                    'Donnas IN', 'Donnas OUT', 'Perloz', 'Sassa', 'Rifugio Coda',
#                                    'Rifugio della Barma', 'Lago Chiaro', 'Col della Vecchia',
#                                    'Niel La Gruba', 'Loo', 'Gressoney IN', 'Gressoney OUT',
#                                    'Rifugio Alpenzu', 'Champoluc', 'Rifugio Grand Tournalin',
#                                    'Valtournenche IN', 'Valtournenche OUT', 'Rifugio Barmasse', 'Vareton',
#                                    'Rifugio Magià', 'Rifugio Cuney', 'Bivacco R. Clermont', 'Oyace',
#                                    'Bruson Arp', 'Ollomont IN', 'Ollomont OUT', 'Rifugio Champillon',
#                                    'Ponteille Desot', 'Bosses', 'Rifugio Frassati', 'Pas Entre Deux Sauts',
#                                    'Monte de la Saxe', 'FINISH'])

# all_aid_station_bib_stats_df

### Who ran too easy or hard at the start?

In [ ]:
# sub_TOR330_dem_bib_list = list(TOR330_dem['PK'][TOR330_dem['Finish Category'] == 'Sub-130'].unique())

# new_bib_list = []
# for bib in sub_TOR330_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']> 14) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche IN') )
#     &
#                 (lifebase_bib_df['PK']== bib)
#                ]
    
#     new_bib_list.append(df)
# df= pd.concat(new_bib_list)
# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK','Lifebase', 'Timestamp', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '*'*40,  '\n',)
#     print(TOR330_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR330_dem['PK'] == pk)  ], '\n','\n','\n','\n', '*'*40,  '\n',)

In [ ]:
# sub_TOR330_dem_bib_list = list(TOR330_dem['PK'][TOR330_dem['Finish Category'] == 'Sub-140'].unique())

# new_bib_list = []
# for bib in sub_TOR330_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']< 10) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche IN') ) &
#                 (lifebase_bib_df['PK']== bib)]
#     new_bib_list.append(df)
    
# df= pd.concat(new_bib_list)

# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK', 'Wave', 'Lifebase', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '\n',)
#     print(TOR330_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR330_dem['PK'] == pk)  ], '\n', '*'*40,  '\n',)

In [ ]:
# # reading in Raw Data
# races = ['TOR330']
# years = [ 
# #     '2021',
# #         '2022',
# #          '2023', 
#     '2024'
#         ]

# TORX_df = {}

# for race in races:
#     for year in years:
#         df = pd.read_excel(f'{race} Data/1. 100x100trail/{race}_{year}.xlsx',
#                                  dtype={'Start Date': 'string',
#                                         'Year': 'string'})
#         print(f'{race}_{year} {df.shape}')
#         # Store the DataFrame in the dictionary with a key like 'TOR330_2021'
#         TORX_df[f'{race}_{year}'] = df
#     print('*'*50)
    
# TORX_df_concat = pd.concat(TORX_df)
# TOR330 = TORX_df_concat[TORX_df_concat['Year'] == year]

In [ ]:
# sub_TOR330_dem_bib_list = list(TOR330_dem['PK'][TOR330_dem['Finish Category'] == 'Sub-90'].unique())

# new_bib_list = []
# for bib in sub_TOR330_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']> 8) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche OUT') ) &
#                 (lifebase_bib_df['PK']== bib)]
#     new_bib_list.append(df)
    
# df= pd.concat(new_bib_list)

# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK', 'Wave', 'Lifebase', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '\n',)
#     print(TOR330_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR330_dem['PK'] == pk)  ], '\n', '*'*40,  '\n',)